# Notebook 05 — Planning: the model writes a plan, our code checks and runs it

**Time:** about 35 minutes

**What you will learn**
- How a model can write a **plan** (a list of steps) instead of answering at once
- Why we **check the plan** before running anything
- How to ask the model to **fix a bad plan** (re-planning), with a limit

### The big idea
In Notebook 03 the model chose one tool at a time. Here it writes the **whole plan first**, as JSON:

```json
{"steps": [{"tool": "get_price",   "product_id": "SG101"},
           {"tool": "check_stock", "product_id": "SG101"}]}
```
Our code then checks every step, runs the steps, and finally the model writes the answer from the **results**.

**Plan → Check → Run → Answer**

### ▶️ Run the two setup cells below first
They check that Ollama is ready and create `ask()`, the one helper we use to talk to the model.

In [1]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

Models on this computer: ['llama3.2:latest']
✅ Ready to go with llama3.2


In [2]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

ask() is ready.


## Part 1 — The tools the plan may use

In [3]:
import pandas as pd

PRODUCTS = pd.read_csv("../datasets/products.csv")
PRODUCT_IDS = list(PRODUCTS["product_id"])

def get_price(product_id):
    row = PRODUCTS[PRODUCTS["product_id"] == product_id]
    return float(row.iloc[0]["price"])

def check_stock(product_id):
    row = PRODUCTS[PRODUCTS["product_id"] == product_id]
    return int(row.iloc[0]["stock"])

TOOLS = {"get_price": get_price, "check_stock": check_stock}
print(get_price("SG101"), check_stock("SG101"))

79.0 12


## Part 2 — Ask the model for a plan (as JSON)
Pydantic classes describe the shape of the plan; Ollama makes the reply follow that shape.

In [4]:
from pydantic import BaseModel

class Step(BaseModel):
    tool: str
    product_id: str

class Plan(BaseModel):
    steps: list[Step]

PLANNER_RULES = """You are a planner. Write a list of steps.
Each step uses ONE tool on ONE product id.
Tools: get_price (price of a product), check_stock (units).
Use only these tools and only product ids from the question."""

In [5]:
def make_plan(question, extra_note=""):
    messages = [{"role": "system", "content": PLANNER_RULES},
                {"role": "user", "content": question + extra_note}]
    reply = ollama.chat(model=MODEL, messages=messages,
                        format=Plan.model_json_schema(),
                        options={"temperature": 0})
    try:
        return Plan.model_validate_json(reply.message.content)
    except Exception:
        return Plan(steps=[])     # unreadable reply → empty plan

QUESTION = ("Which is cheaper, SG101 or SG104, "
            "and is the cheaper one in stock?")
plan = make_plan(QUESTION)
for number, step in enumerate(plan.steps, 1):
    print(number, step.tool, step.product_id)

1 get_price SG101
2 get_price SG104
3 check_stock SG101
4 check_stock SG104


## Part 3 — Check the plan BEFORE running it
A plan from a model can be wrong: an unknown tool, a product that does not exist, or far too many steps.

In [6]:
def check_plan(plan):
    problems = []
    if len(plan.steps) == 0:
        problems.append("The plan has no steps.")
    if len(plan.steps) > 8:
        problems.append("The plan has more than 8 steps.")
    for number, step in enumerate(plan.steps, 1):
        if step.tool not in TOOLS:
            problems.append(f"Step {number}: unknown tool "
                            f"'{step.tool}'.")
        if step.product_id not in PRODUCT_IDS:
            problems.append(f"Step {number}: unknown product "
                            f"'{step.product_id}'.")
    return problems

problems = check_plan(plan)
print("Problems:", problems or "none — the plan looks fine")

Problems: none — the plan looks fine


## Part 4 — Re-planning: send the problems back (at most 2 times)

In [7]:
for attempt in range(2):                   # at most 2 fixes
    if not problems:
        break
    print("Asking the model to fix the plan...")
    note = "\n\nYour last plan had these problems: "
    note += " ".join(problems)
    plan = make_plan(QUESTION, extra_note=note)
    problems = check_plan(plan)

if problems:
    print("❌ The plan is still not valid:", problems)
    print("   Try again, or use a bigger model (qwen2.5:7b).")
else:
    steps = [(s.tool, s.product_id) for s in plan.steps]
    print("✅ Plan accepted:", steps)

✅ Plan accepted: [('get_price', 'SG101'), ('get_price', 'SG104'), ('check_stock', 'SG101'), ('check_stock', 'SG104')]


## Part 5 — Run the plan, then write the answer from the results

In [8]:
results = []
if not problems:
    for step in plan.steps:
        value = TOOLS[step.tool](step.product_id)
        results.append(f"{step.tool}({step.product_id}) = {value}")
        print("✔", results[-1])

if results:
    facts = "\n".join(results)
    answer = ask(f"Question: {QUESTION}\n\nFacts:\n{facts}\n\n"
                 "Answer using ONLY these facts.")
    print("\nANSWER:", answer)

✔ get_price(SG101) = 79.0
✔ get_price(SG104) = 89.0
✔ check_stock(SG101) = 12
✔ check_stock(SG104) = 18

ANSWER: SG101 is cheaper, at $79.00. It is also in stock, with 12 units available.


### Check against the truth

In [9]:
check = PRODUCTS["product_id"].isin(["SG101", "SG104"])
PRODUCTS[check][["product_id", "name", "price", "stock"]]

,product_id,name,price,stock
0,SG101,Classic Round,79.0,12
3,SG104,Urban Square,89.0,18


## 💬 Discuss
- What could go wrong if we ran a plan **without** checking it first?
- Why does the final answer use only the **results**, not the model's memory?
- Would you let a planning agent book and pay for a flight without asking you? Why not?

## ✍️ Try it yourself
1. Change `QUESTION` to: *"Compare the prices of SG105, SG106 and SG107. Which ones are in stock?"*
2. Add a rule to `check_plan`: two identical steps are not allowed.

## ✅ Summary
- **Plan → Check → Run → Answer.**
- The model writes the plan; **our code** checks it and runs it.
- Bad plans are sent back with the problems, but only a limited number of times.

**Next:** Notebook 06 — several agents working as a **team**.